<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_040_Estructuras_iterativas_series_de_pacientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 40. Estructuras iterativas para el procesamiento de series de pacientes</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 1. Descubriendo Python**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Práctica:** [unidad 40 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

**Para dar la clase con el cuaderno:** usa **Entorno de ejecución → Ejecutar todo** antes de empezar, para que se dibujen los gráficos. Las celdas con borde azul son las diapositivas; el desarrollo de cada bloque está debajo y se puede plegar con la flecha de su título. El índice (☰, a la izquierda) sirve para saltar de un bloque a otro.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Recorrer una serie de mediciones o una cohorte de pacientes con un bucle `for`.
2. Usar acumuladores y contadores para calcular totales, medias y proporciones.
3. Combinar bucles y condiciones para detectar valores de alarma.
4. Recorrer varias listas a la vez con `zip()` y numerar los elementos con `enumerate()`.
5. Repetir un cálculo mientras se cumpla una condición con un bucle `while`.
6. Calcular la prevalencia de una condición en una cohorte sintética.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: bucles con datos clínicos | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Ocho pacientes, 40 glucemias: ¿quién ha tenido hipoglucemia?</p><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Revisar a mano cada valor es lento y propenso a errores, y mañana habrá que repetirlo.</li><li>La instrucción es siempre la misma: <b>para cada</b> paciente y <b>para cada</b> glucemia, comprueba si es < 70.</li></ul></div>

In [ ]:
#@title Figura: Ocho pacientes, 40 glucemias: ¿quién ha tenido hipoglucemia?
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(40)
plt.figure(figsize=(10, 4.5))
for p in range(8):
    g = rng.normal(140, 45, rng.integers(4, 7)).clip(45, 300)
    plt.scatter([p + 1] * len(g), g, s=90, c=["firebrick" if x < 70 else "steelblue" for x in g], zorder=3)
plt.axhline(70, color="firebrick", linestyle="--", label="Hipoglucemia: < 70 mg/dL")
plt.xticks(range(1, 9), [f"P{p}" for p in range(1, 9)])
plt.xlabel("Paciente")
plt.ylabel("Glucemia capilar (mg/dL)")
plt.title("Glucemias de las últimas 24 horas", fontweight="bold")
plt.legend()
plt.show()

#### Desarrollo para el estudio

La supervisora de una planta de Endocrinología pide un informe al final del turno: de los 8 pacientes con diabetes ingresados, ¿cuántos han tenido alguna **hipoglucemia** (glucemia < 70 mg/dL) en las últimas 24 horas, y cuál ha sido la glucemia media de cada uno?

Cada paciente tiene entre 4 y 6 glucemias capilares. Revisarlas una a una es lento y propenso a errores, y mañana habrá que repetirlo. Lo que se necesita es una instrucción que diga: **para cada paciente**, y **para cada glucemia** de ese paciente, comprueba si es menor de 70.

Repetir la misma operación sobre todos los elementos de una serie es lo que hace un **bucle**, y es la base de cualquier análisis de datos.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 El bucle for</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">El bucle for recorre la serie elemento a elemento</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_040_for.png" width="780" alt="El bucle for recorre la serie elemento a elemento"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>En cada vuelta, la variable toma el siguiente valor de la lista.</li><li>El cuerpo del bucle (sangrado) se repite tantas veces como elementos haya.</li></ul></div>

#### Desarrollo para el estudio

```python
for elemento in serie:
    acciones con elemento
```

- `serie` puede ser una lista, una cadena, un `range()` o cualquier colección.
- En cada vuelta (**iteración**), `elemento` toma el siguiente valor de la serie.
- Las acciones van sangradas, como en un `if`.

### <font color='steelblue'>2.2 Acumuladores y contadores</font>

Para resumir una serie se usa una variable que se inicializa **antes** del bucle y se actualiza **dentro** de él:

| Patrón | Inicialización | Dentro del bucle | Ejemplo clínico |
| --- | --- | --- | --- |
| Acumulador | `total = 0` | `total = total + valor` | Suma de glucemias para la media |
| Contador | `n = 0` | `n = n + 1` si se cumple una condición | Número de hipoglucemias |
| Lista de resultados | `resultados = []` | `resultados.append(...)` | IMC de cada paciente |

`total = total + valor` puede abreviarse como `total += valor`.

### <font color='steelblue'>2.3 El bucle while</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">while: repetir mientras se cumpla una condición</p><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Se usa cuando no se sabe de antemano cuántas vueltas hacen falta.</li><li>Cuidado: si la condición nunca deja de cumplirse, el bucle no termina.</li></ul></div>

In [ ]:
#@title Figura: while: repetir mientras se cumpla una condición
import matplotlib.pyplot as plt

concentracion, umbral, horas, t = 16.0, 1.0, [0], [16.0]
while concentracion >= umbral:          # se repite mientras quede fármaco por encima del umbral
    concentracion = concentracion / 2   # una semivida
    horas.append(horas[-1] + 6)
    t.append(concentracion)
plt.figure(figsize=(10, 4.5))
plt.plot(horas, t, "o-", color="steelblue", linewidth=2.5, markersize=9)
plt.axhline(umbral, color="firebrick", linestyle="--", label="Umbral: el bucle se detiene")
for h, c in zip(horas, t):
    plt.annotate(f"{c:g}", (h, c), textcoords="offset points", xytext=(0, 10), ha="center")
plt.xlabel("Horas (semivida de 6 h)")
plt.ylabel("Concentración (mg/L)")
plt.title(f"El bucle dio {len(horas) - 1} vueltas", fontweight="bold")
plt.legend()
plt.show()

#### Desarrollo para el estudio

```python
while condición:
    acciones
```

Repite las acciones **mientras** la condición sea verdadera. Se usa cuando no se sabe de antemano cuántas repeticiones harán falta, por ejemplo, cuántas vidas medias tarda un fármaco en bajar de un nivel. Dentro del bucle algo tiene que cambiar para que la condición acabe siendo falsa; si no, el bucle no termina nunca.

### <font color='steelblue'>2.4 Prevalencia</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Prevalencia = casos / total</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_040_prevalencia.png" width="780" alt="Prevalencia = casos / total"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Con un contador dentro de un bucle se calcula la primera medida de frecuencia.</li><li>Es una proporción: entre 0 y 1, o entre 0 % y 100 %, y siempre en un momento dado.</li></ul></div>

#### Desarrollo para el estudio

La **prevalencia** es la proporción de individuos de una población que presentan una condición en un momento dado:

$$\text{Prevalencia} = \frac{\text{número de casos}}{\text{número total de individuos}}$$

Es el primer indicador que calcularemos con un bucle: un contador de casos dividido entre el total.

## <font color='steelblue'>3. Demostración: bucles con datos clínicos (20 min)</font>

### <font color='steelblue'>3.1 Recorrer una serie de mediciones</font>

In [ ]:
glucemias_mg_dl = [182, 165, 64, 148, 139, 92]

for glucemia in glucemias_mg_dl:
    print(glucemia, "mg/dL")

### <font color='steelblue'>3.2 Acumulador: la media</font>

In [ ]:
total = 0
for glucemia in glucemias_mg_dl:
    total += glucemia

media = total / len(glucemias_mg_dl)
print(f"Glucemia media: {media:.1f} mg/dL")

Python tiene la función `sum()`, que hace lo mismo en una línea. Escribir el bucle sirve para entender qué hace `sum()` por dentro y para cuando el cálculo es más complejo que una suma.

In [ ]:
print(sum(glucemias_mg_dl) / len(glucemias_mg_dl))

### <font color='steelblue'>3.3 Bucle con condición: detectar hipoglucemias</font>

In [ ]:
hipoglucemias = 0
for glucemia in glucemias_mg_dl:
    if glucemia < 70:
        hipoglucemias += 1
        print(f"Hipoglucemia detectada: {glucemia} mg/dL")

print("Total de hipoglucemias:", hipoglucemias)

### <font color='steelblue'>3.4 Numerar con enumerate() y emparejar con zip()</font>

`enumerate()` da a la vez la posición y el valor. Aquí se suma 1 a la posición para numerar las tomas desde 1, como se haría en la gráfica de enfermería.

In [ ]:
for posicion, glucemia in enumerate(glucemias_mg_dl):
    print(f"Toma {posicion + 1}: {glucemia} mg/dL")

`zip()` recorre varias listas a la vez, emparejando sus elementos. Útil cuando sistólica y diastólica se guardan en listas separadas.

In [ ]:
horas = ["08:00", "14:00", "20:00"]
pas_mmHg = [152, 138, 145]
pad_mmHg = [94, 86, 91]

for hora, pas, pad in zip(horas, pas_mmHg, pad_mmHg):
    pam = pad + (pas - pad) / 3
    print(f"{hora}  PA {pas}/{pad} mmHg  PAM {pam:.0f} mmHg")

### <font color='steelblue'>3.5 Repetir un número de veces: range()</font>

`range(n)` genera los números de 0 a n − 1. `range(inicio, fin, paso)` permite elegir el inicio, el final (no incluido) y el salto.

In [ ]:
# Horas de administración de un antibiótico cada 8 horas durante un día
for hora in range(0, 24, 8):
    print(f"Dosis a las {hora:02d}:00")

### <font color='steelblue'>3.6 Recorrer una cohorte</font>

La cohorte del caso: cada paciente es un diccionario con su código y sus glucemias. Un bucle **dentro de otro** (bucles anidados) recorre primero los pacientes y, para cada uno, sus glucemias.

In [ ]:
cohorte = [
    {"codigo": "E-01", "glucemias": [182, 165, 148, 139]},
    {"codigo": "E-02", "glucemias": [96, 64, 110, 128, 105]},
    {"codigo": "E-03", "glucemias": [210, 198, 176, 160, 155, 149]},
    {"codigo": "E-04", "glucemias": [88, 72, 69, 91]},
    {"codigo": "E-05", "glucemias": [140, 132, 151, 147]},
    {"codigo": "E-06", "glucemias": [120, 115, 98, 101, 110]},
    {"codigo": "E-07", "glucemias": [65, 58, 92, 140]},
    {"codigo": "E-08", "glucemias": [170, 162, 158, 149, 152]},
]

pacientes_con_hipoglucemia = 0

for paciente in cohorte:
    glucemias = paciente["glucemias"]
    media = sum(glucemias) / len(glucemias)

    tuvo_hipoglucemia = False
    for glucemia in glucemias:
        if glucemia < 70:
            tuvo_hipoglucemia = True

    if tuvo_hipoglucemia:
        pacientes_con_hipoglucemia += 1

    marca = "  ← hipoglucemia" if tuvo_hipoglucemia else ""
    print(f"{paciente['codigo']}: media {media:.0f} mg/dL{marca}")

proporcion = pacientes_con_hipoglucemia / len(cohorte)
print(f"\nPacientes con alguna hipoglucemia: {pacientes_con_hipoglucemia} de {len(cohorte)} ({proporcion:.0%})")

El formato `{proporcion:.0%}` multiplica por 100 y añade el signo de porcentaje.

### <font color='steelblue'>3.7 El bucle while: eliminación de un fármaco</font>

Un fármaco con una **vida media** de 6 horas reduce su concentración plasmática a la mitad cada 6 horas. Si la concentración inicial es 40 mg/L, ¿cuántas horas tardará en bajar a 5 mg/L o menos? No sabemos de antemano cuántas vueltas hará falta dar: es un caso típico de `while`.

In [ ]:
concentracion_mg_l = 40
vida_media_h = 6
umbral_mg_l = 5
horas = 0

while concentracion_mg_l > umbral_mg_l:
    concentracion_mg_l = concentracion_mg_l / 2
    horas += vida_media_h
    print(f"A las {horas} h: {concentracion_mg_l:.1f} mg/L")

print(f"Tarda {horas} horas ({horas // vida_media_h} vidas medias) en llegar a {umbral_mg_l} mg/L")

### <font color='steelblue'>Ampliación: break y continue</font>

`break` sale del bucle en cuanto se cumple algo; `continue` salta a la siguiente vuelta. Útil para detenerse en la **primera** alarma o para ignorar datos ausentes.

In [ ]:
saturaciones = [97, 96, None, 94, 89, 95]

for toma, sat in enumerate(saturaciones, start=1):
    if sat is None:
        print(f"Toma {toma}: sin dato, se omite")
        continue
    if sat < 90:
        print(f"Toma {toma}: saturación {sat} %. Avisar al médico de guardia.")
        break
    print(f"Toma {toma}: {sat} %")

### <font color='steelblue'>Ampliación: listas por comprensión</font>

Una **lista por comprensión** construye una lista nueva en una sola línea a partir de otra. Es una forma compacta de escribir un bucle con `append()`.

In [ ]:
glucemias_mg_dl = [182, 165, 64, 148, 139, 92]

glucemias_mmol_l = [round(g / 18, 1) for g in glucemias_mg_dl]
hipoglucemias = [g for g in glucemias_mg_dl if g < 70]

print(glucemias_mmol_l)
print(hipoglucemias)

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Inicializar el acumulador dentro del bucle.** Si `total = 0` se escribe dentro del `for`, se pone a cero en cada vuelta y al final solo contiene el último valor.

**Olvidar la sangría.** Lo que no está sangrado no forma parte del bucle y se ejecuta una sola vez al final.

**Bucles while infinitos.** Si dentro del bucle no cambia nada que afecte a la condición, el bucle no termina. En Colab se detiene con el botón ■ de la celda.

**Modificar la lista que se recorre.** Añadir o quitar elementos de una lista mientras se recorre con `for` produce resultados imprevisibles; es mejor construir una lista nueva.

**Dividir entre cero.** Al calcular una media o una proporción, comprueba que la serie no está vacía: `len(lista) == 0` produce `ZeroDivisionError`.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 40 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** predecir cuántas veces se ejecuta un bucle y qué valor final tiene un acumulador.
2. **Misión clínica (con pistas):** calcular la prevalencia de hipertensión en una cohorte sintética y el IMC de cada paciente.
3. **Misión reto (con asistente):** programar con `while` cuántas dosis de mantenimiento hacen falta para alcanzar el estado estacionario de un fármaco, y verificar el código que proponga el asistente.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': '¿Cuántas veces se ejecuta print en: for h in range(0, 24, 6): print(h)?',
  'type': 'multiple_choice',
  'answers': [{'answer': '4',
               'correct': True,
               'feedback': 'Correcto: range(0, 24, 6) genera 0, 6, 12 y 18; el 24 no se incluye.'},
              {'answer': '5',
               'correct': False,
               'feedback': 'El final de range no se incluye: 24 no aparece.'},
              {'answer': '24', 'correct': False, 'feedback': 'El paso es 6, no 1.'},
              {'answer': '6',
               'correct': False,
               'feedback': 'Revisa qué números genera range con paso 6.'}]},
 {'question': 'Se quiere contar las glucemias menores de 70 mg/dL. ¿Dónde debe ir contador = 0?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Antes del bucle',
               'correct': True,
               'feedback': 'Correcto: si se pone dentro, el contador vuelve a cero en cada '
                           'vuelta.'},
              {'answer': 'Dentro del bucle, antes del if',
               'correct': False,
               'feedback': 'Se reiniciaría en cada vuelta y solo contaría la última glucemia.'},
              {'answer': 'Dentro del if',
               'correct': False,
               'feedback': 'Se pondría a cero cada vez que se detectara un caso.'},
              {'answer': 'Después del bucle',
               'correct': False,
               'feedback': 'El bucle intentaría usar una variable que todavía no existe.'}]},
 {'question': 'En una cohorte de 250 pacientes, 45 tienen diabetes. ¿Cuál es la prevalencia de '
              'diabetes?',
  'type': 'multiple_choice',
  'answers': [{'answer': '18 %', 'correct': True, 'feedback': 'Correcto: 45 / 250 = 0,18.'},
              {'answer': '45 %',
               'correct': False,
               'feedback': '45 es el número de casos, no la proporción.'},
              {'answer': '5,6 %',
               'correct': False,
               'feedback': 'Sale de dividir 250 entre 45: el orden de la división está invertido.'},
              {'answer': '82 %',
               'correct': False,
               'feedback': 'Esa es la proporción de pacientes sin diabetes.'}]},
 {'question': '¿Cuándo es preferible un bucle while a un bucle for?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Cuando no se sabe de antemano cuántas repeticiones hacen falta',
               'correct': True,
               'feedback': 'Correcto: while repite mientras se cumpla una condición.'},
              {'answer': 'Cuando se recorre una lista de pacientes',
               'correct': False,
               'feedback': 'Para recorrer una lista conocida es más claro un for.'},
              {'answer': 'Siempre, porque es más rápido',
               'correct': False,
               'feedback': 'No es más rápido; cada uno tiene su uso.'},
              {'answer': 'Nunca: while está desaconsejado en Python',
               'correct': False,
               'feedback': 'while es una estructura básica y útil.'}]},
 {'question': 'Un fármaco tiene una vida media de 4 horas y una concentración inicial de 80 mg/L. '
              '¿Qué concentración tiene a las 12 horas?',
  'type': 'multiple_choice',
  'answers': [{'answer': '10 mg/L',
               'correct': True,
               'feedback': 'Correcto: 12 horas son 3 vidas medias: 80 → 40 → 20 → 10.'},
              {'answer': '20 mg/L',
               'correct': False,
               'feedback': 'Eso son solo 2 vidas medias (8 horas).'},
              {'answer': '26,7 mg/L',
               'correct': False,
               'feedback': 'La concentración no baja en proporción lineal: se reduce a la mitad en '
                           'cada vida media.'},
              {'answer': '0 mg/L',
               'correct': False,
               'feedback': 'La concentración se reduce a la mitad en cada vida media, pero nunca '
                           'llega exactamente a cero.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- Un bucle `for` aplica la misma operación a cada elemento de una serie.
- Los acumuladores y contadores se inicializan antes del bucle y se actualizan dentro.
- Un bucle con un `if` dentro detecta valores de alarma y calcula proporciones como la prevalencia.
- `while` repite mientras se cumpla una condición; dentro, algo debe cambiar para que termine.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Bucle | Estructura que repite un bloque de instrucciones |
| Iteración | Cada una de las vueltas de un bucle |
| Acumulador | Variable que va sumando valores en cada iteración |
| Contador | Variable que cuenta cuántas veces se cumple una condición |
| Bucles anidados | Un bucle dentro de otro |
| Prevalencia | Proporción de individuos con una condición en un momento dado |
| Vida media | Tiempo que tarda la concentración de un fármaco en reducirse a la mitad |

## <font color='steelblue'>Para saber más</font>

- [Tutorial de Python: la sentencia for y la función range()](https://docs.python.org/es/3/tutorial/controlflow.html#for-statements)
- [Tutorial de Python: listas por comprensión](https://docs.python.org/es/3/tutorial/datastructures.html#list-comprehensions)

---
*Tema 40 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*